In [ ]:
from __future__ import absolute_import, division, print_function, unicode_literals # Import future features for Python 2/3 compatibility
import tensorflow as tf # Import TensorFlow for machine learning
import pandas as pd # Import pandas for data manipulation and analysis

In [ ]:
CSV_COLUMN_NAMES= ['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth', 'Species']
SPECIES= ['Setosa', 'Versicolor', 'Virginica']


In [ ]:
'''
 uses TensorFlow's utility function to download dataset files:
 The lines download the Iris training and testing dataset (a CSV file) from the specified Google Cloud Storage URL.
 The downloaded files will be saved locally as iris_training.csv and iris_testing.csv respectively, and its local path is stored in a variable.
'''
train_path = tf.keras.utils.get_file(
    "iris_training.csv", "https://storage.googleapis.com/download.tensorflow.org/data/iris_training.csv")
test_path = tf.keras.utils.get_file(
    "iris_test.csv", "https://storage.googleapis.com/download.tensorflow.org/data/iris_test.csv")

In [ ]:
''''
Loading dataframes using pandas
pd.read_csv() is used to read the CSV files into pandas dataframes.
paths show the direction of files
names is for column names
header=0 takes row 0 as header
'''

train_df = pd.read_csv(train_path, names=CSV_COLUMN_NAMES, header=0)
test_df = pd.read_csv(test_path, names=CSV_COLUMN_NAMES, header=0)

In [ ]:
train_df.head()#first few rows of training dataset

,SepalLength,SepalWidth,PetalLength,PetalWidth,Species
0,6.4,2.8,5.6,2.2,2
1,5.0,2.3,3.3,1.0,1
2,4.9,2.5,4.5,1.7,2
3,4.9,3.1,1.5,0.1,0
4,5.7,3.8,1.7,0.3,0


In [ ]:
#target labels
y_train=train_df.pop('Species')
y_test=test_df.pop('Species')

In [ ]:
'''
shape of dataset
4 columns and 120 entries
'''
train_df.shape

(120, 4)

In [ ]:
'''
1. DATA TRANSFORMATION BRIDGE:
   - Converts static data (Pandas/NumPy) into dynamic TensorFlow
     Dataset objects
   - Enables efficient streaming of data during training
   - Memory-efficient processing for large datasets

2. TRAINING PERFORMANCE OPTIMIZATION:
   - Shuffling prevents model from learning data order patterns
   - Batching enables parallel processing on GPU/TPU
   - Repeat() creates infinite dataset for multi-epoch training

   - Training mode: Shuffled, repeated, batched for learning
'''
def input_fn(features,labels,training=True,batch_size=256):
  ds=tf.Data.Dataset.from_tensor_slices((dict(features),labels))
  if training:
    ds=ds.shuffle(1000).repeat() #repeat shuffling indefinitely
  return ds.batch(batch_size)


In [ ]:

'''
feauture columns tells the model how to handle the input
without it the model wouldnt know whats contained within each column
when we call numeric column it knows that this is indeed a number
Convert input to 32-bit floating point numbers
Standard for ML: Most neural networks use float32
'''
my_feature_columns=[]
for key in train_df.keys():
  my_feature_columns.append(tf.feature_column.numeric_column(key=key))
print(my_feature_columns)

[NumericColumn(key='SepalLength', shape=(1,), default_value=None, dtype=tf.float32, normalizer_fn=None), NumericColumn(key='SepalWidth', shape=(1,), default_value=None, dtype=tf.float32, normalizer_fn=None), NumericColumn(key='PetalLength', shape=(1,), default_value=None, dtype=tf.float32, normalizer_fn=None), NumericColumn(key='PetalWidth', shape=(1,), default_value=None, dtype=tf.float32, normalizer_fn=None)]


In [ ]:
'''
Deep Neural Network Classifier(DNN)
Feature Columns
-Bridge between raw data and model
-Numeric, categorical, bucketized, crossed columns

Model Architecture
-we have 2 hidden layers
-layer 1 has 30 neurons
-layer 2 has 10 neurons
-output layer has 3 neurons(its supposed to give us either of 3 predictions)
This Version is outdated though it has moved to tensorflow keras
'''



classifier = tf.estimator.DNNClassifier(feature_columns=my_feature_columns,hidden_units = [30.10],n_classes=3)

AttributeError: module 'tensorflow' has no attribute 'estimator'

In [ ]:
'''
Lambda creates an anonymous function
essential as the train() expects a function object not a ds object
main reason for creating outer and inner fn in last titanic classifier model
estimators have in built triggers that call the input fn needed
input fn returns a fresh dataset each time its called
steps refers to exactly how many times updates its weights
Therefore learning
refers to one forward pass + one backward pass of batches processed
Each step involves processing a batch of data, calculating the loss, and updating the model's parameters (weights and biases) to minimize that loss.
model_dir is the storage of the model
'''
classifier.train(input_fn=lambda : input_fn(train_df,y_train,training=True), model_dir='./iris_model' ,steps=5000)

NameError: name 'classifier' is not defined

In [ ]:
'''
evaluating models perfomance
training is false as testing data is not shuffled
since no learning occurs
also called on a lambda as evaluate() a function
'''

classifier.evaluate(input_fn=lambda : input_fn(test_df,y_test,training=False))


NameError: name 'classifier' is not defined

In [ ]:
'''
creating user interface
classifier.predict() returns a generator
for memory efficiency
the main reason for having a input_fn in prediction stage is
# Minimal prediction pipeline:
1. Convert data → TensorFlow tensors
2. Batch it (optional but efficient)
3. Feed to model

'''
def input_fn(features, batch_size=256):
    # Convert the inputs to a Dataset without labels.
    return tf.data.Dataset.from_tensor_slices(dict(features)).batch(batch_size)

features = ['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth']
predict = {}

print("Please type numeric values as prompted.")
for feature in features:
    valid = True
    while valid:
        val = input(feature + ": ")  # collecting inputs from user
        if not val.isdigit(): #not so as to input floating value
            valid = False
            predict[feature] = [float(val)]  # Loading the predict dictionary with users inputs

predictions = classifier.predict(input_fn=lambda: input_fn(predict))# .predict() expects a functions

for pred_dict in predictions:  # looping through predictions, returns class_ids & probabilities
    class_id = pred_dict['class_ids'][0]  # [0] extracts value (only 1 item in the list)
    probability = pred_dict['probabilities'][class_id]
    print(f'Prediction is "{SPECIES[class_id]} ({100 * probability:.1f}%)' )